In [1]:
from google.colab import drive
import pandas as pd
import numpy as np
import torchaudio
import zipfile
import ast
import os

In [2]:
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
print(f"torchaudio version: {torchaudio.__version__}")


pandas version: 2.2.2
numpy version: 2.0.2
torchaudio version: 2.11.0+cpu


In [3]:
drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/dataset.zip"
extract_path = "/content/dataset/"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP başarıyla çıkarıldı")

test_csv = '/content/drive/MyDrive/SSL_Projesi/test_metadata.csv'
df = pd.read_csv(test_csv)

print(df.head())
print("Toplam test örneği:", len(df))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ZIP başarıyla çıkarıldı
                                          audio_path  azimuth_deg  distance_m  \
0  /content/dataset/dataset/session_1775030812_5c...    79.932860    2.810450   
1  /content/dataset/dataset/session_1775040605_e2...   262.116227    4.355966   
2  /content/dataset/dataset/session_1775040766_d2...    70.257374    3.220559   
3  /content/dataset/dataset/session_1775030073_23...   234.963847    3.535452   
4  /content/dataset/dataset/session_1775041687_e1...   259.779367    2.567600   

      pos_x     pos_y  pos_z  snr_db  ambient_noise_db  rt60 room_dimension  
0  2.991272  4.900000    1.5      30                15   0.3      (5, 5, 3)  
1  4.402518  0.685205    2.5       0                 0   0.3    (10, 10, 5)  
2  3.587891  4.900000    1.5      30                 0   0.5      (5, 5, 3)  
3  0.470321  0.100000    1.5      15            

In [4]:
radius = 0.1

mics = []

for i in range(7):
    angle = 2 * np.pi * i / 7
    x = radius * np.cos(angle)
    y = radius * np.sin(angle)
    mics.append([x, y, 0])

mics.append([0, 0, 0])

mic_locs = np.array(mics).T

Frame based gcc

In [5]:
def parabolic_interpolation(cc, idx):
    if idx <= 0 or idx >= len(cc) - 1:
        return idx

    y1 = cc[idx - 1]
    y2 = cc[idx]
    y3 = cc[idx + 1]

    denom = (y1 - 2*y2 + y3)
    if abs(denom) < 1e-12:
        return idx

    delta = 0.5 * (y1 - y3) / denom
    return idx + delta

In [6]:
def stft_gcc_phat(sig1, sig2, fs=16000, n_fft=1024, hop_length=512):
    eps = 1e-8

    window = np.hanning(n_fft)

    n_frames = 1 + (len(sig1) - n_fft) // hop_length

    R_accum = np.zeros(n_fft//2 + 1, dtype=np.complex64)

    for i in range(n_frames):
        start = i * hop_length

        frame1 = sig1[start:start+n_fft] * window
        frame2 = sig2[start:start+n_fft] * window

        X1 = np.fft.rfft(frame1, n=n_fft)
        X2 = np.fft.rfft(frame2, n=n_fft)

        # cross spectrum
        R = X1 * np.conj(X2)

        # PHAT
        #R = R / (np.abs(R) + eps)

        R_accum += R

    R_accum /= n_frames

    # inverse FFT → correlation
    cc = np.fft.irfft(R_accum, n=n_fft)

    # center shift
    max_shift = n_fft // 2
    cc = np.concatenate((cc[-max_shift:], cc[:max_shift]))

    # integer peak
    idx = np.argmax(np.abs(cc))

    # 🔥 fractional refinement
    frac_idx = parabolic_interpolation(np.abs(cc), idx)

    # convert to shift
    shift = frac_idx - max_shift

    tau = shift / fs

    return tau, cc

In [7]:
def compute_tdoas(signals, fs, ref_idx):
  M = signals.shape[0]
  tdoas = {}

  for i in range(M):
    if i == ref_idx:
      continue

    tau, cc = stft_gcc_phat(signals[i], signals[ref_idx], fs)

    tdoas[i] = tau

  return tdoas

In [42]:
def estimate_azimuth(signals, mic_locs, fs, ref_idx):
    c = 343.0
    tdoas = compute_tdoas(signals, fs, ref_idx)

    taus = []
    mic_vecs = []

    for i, tau in tdoas.items():
        mic_vecs.append(mic_locs[:, i] - mic_locs[:, ref_idx])
        taus.append(tau)

    mic_vecs = np.array(mic_vecs)[:, :2]
    taus = np.array(taus)

    angles = np.linspace(0, 2*np.pi, 360)
    scores = []

    for th in angles:
        u = np.array([np.cos(th), np.sin(th)])
        tau_pred = (mic_vecs @ u) / c
        score = -np.mean((taus - tau_pred)**2)
        scores.append(score)

    scores = np.array(scores)

    # --- 1. Tahmin: En hassas değer (Değişmedi) ---
    best_idx = np.argmax(scores)
    estimated_angle = (np.degrees(angles[best_idx]) + 180) % 360

    # --- 2. Çizim için Spektrum: 'scores' üzerinden Gaussian yumuşatma ---
    # Bu yöntem parabolik çukuru, makale standartlarında bir "Lobe" (dağ) yapısına çevirir.
    # Sigma değerini 1e-5 ile 1e-4 arasında değiştirerek tepenin genişliğini ayarlayabilirsin.
    sigma = 5e-5
    spatial_spectrum = np.exp(scores / sigma)

    # Normalize et (0 ile 1 arası)
    spatial_spectrum = (spatial_spectrum - np.min(spatial_spectrum)) / (np.max(spatial_spectrum) - np.min(spatial_spectrum) + 1e-12)

    shifted_angles = (np.degrees(angles) + 180) % 360

    # Sıralama işlemleri
    sort_idx = np.argsort(shifted_angles)
    final_angles = shifted_angles[sort_idx]
    final_spectrum = spatial_spectrum[sort_idx]

    # Sadece tek bir return
    return estimated_angle, final_spectrum, final_angles

In [50]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

def save_combined_spectrum_plot(
    grid_angles,
    spec_success, true_success, est_success,
    spec_fail, true_fail, est_fail,
    filename="/content/music_karsilastirma.png"
):
    # 1. MAKALE STANDARTLARI İÇİN SEABORN AYARLARI
    sns.set_theme(style="ticks", context="paper", font_scale=1.2)
    plt.rcParams["font.family"] = "serif"

    # 1 satır, 2 sütunlu geniş bir figür oluştur
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # --- Yeni Akademik Renk Paleti ---
    color_spectrum = "#003366"  # Tok Akademik Lacivert
    color_true = "#d7191c"      # Dikkat çekici Kırmızı
    color_est = "#000000"       # Siyah (Kesik çizgilerde en iyi kontrast)

    # ==========================================
    # (a) BAŞARILI SENARYO (Sol Grafik)
    # ==========================================
    ax1 = axes[0]
    spec_success_norm = spec_success / np.max(spec_success)

    sns.lineplot(x=grid_angles, y=spec_success_norm, ax=ax1,
                 label="Psödo-Spektrum $P(\\theta)$", color=color_spectrum, linewidth=2)
    ax1.axvline(x=true_success, color=color_true, linestyle='--', linewidth=2.5,
                label=f"Gerçek Açı ({true_success:.1f}°)")
    ax1.axvline(x=est_success, color=color_est, linestyle=':', linewidth=2.5,
                label=f"Tahmin ({est_success:.1f}°)")

    ax1.set_xlim([0, 360])
    ax1.set_ylim([0, 1.05])
    ax1.set_xlabel("Arama Açısı (Derece)", fontweight='bold')
    ax1.set_ylabel("Normalize Psödo-Spektrum", fontweight='bold')
    ax1.set_title("(a) Başarılı Kestirim", pad=15, fontweight='bold')

    # Lejant sağ alta taşındı
    ax1.legend(loc="lower right", frameon=True, edgecolor='black')
    ax1.grid(axis='y', linestyle=':', alpha=0.6)

    # ==========================================
    # (b) HATALI SENARYO (Sağ Grafik)
    # ==========================================
    ax2 = axes[1]
    spec_fail_norm = spec_fail / np.max(spec_fail)

    sns.lineplot(x=grid_angles, y=spec_fail_norm, ax=ax2,
                 label="Psödo-Spektrum $P(\\theta)$", color=color_spectrum, linewidth=2)
    ax2.axvline(x=true_fail, color=color_true, linestyle='--', linewidth=2.5,
                label=f"Gerçek Açı ({true_fail:.1f}°)")
    ax2.axvline(x=est_fail, color=color_est, linestyle=':', linewidth=2.5,
                label=f"Tahmin ({est_fail:.1f}°)")

    ax2.set_xlim([0, 360])
    ax2.set_ylim([0, 1.05])
    ax2.set_xlabel("Arama Açısı (Derece)", fontweight='bold')
    ax2.set_ylabel("Normalize Psödo-Spektrum", fontweight='bold')
    ax2.set_title("(b) Hatalı Kestirim", pad=15, fontweight='bold')

    # Lejant sağ alta taşındı
    ax2.legend(loc="lower right", frameon=True, edgecolor='black')
    ax2.grid(axis='y', linestyle=':', alpha=0.6)

    # ==========================================
    # SON RÖTUŞLAR VE KAYDETME
    # ==========================================
    sns.despine(fig=fig, top=True, right=True)
    plt.tight_layout()

    plt.savefig(filename, dpi=600, bbox_inches='tight')
    plt.close()
    print(f"✅ Karşılaştırmalı grafik başarıyla kaydedildi: {filename}")

In [51]:
results = []
plot_data = []
plots_dir = "/content/"
os.makedirs(plots_dir, exist_ok=True)

col_file = "audio_path" if "audio_path" in df.columns else "file"
col_true = "azimuth_deg" if "azimuth_deg" in df.columns else "true_angle"

hedef_kelimeler = r"1775041771_9043c1/sample_00003\.wav|1775030993_77e7d1/sample_00002\.wav"
test_df = df[df[col_file].str.contains(hedef_kelimeler, na=False, regex=True)].reset_index(drop=True)

# Saf Python Sözlüğü (Çökme Koruması)
records = test_df.to_dict(orient="records")
print(f"GCC-PHAT İçin İşlenecek Dosya Sayısı: {len(records)}")

# ==========================================
# 3. ÇÖKMEYEN İŞLEM DÖNGÜSÜ
# ==========================================
for i, row in enumerate(records):
    rel_path = row[col_file]
    true_angle = row[col_true]

    wav_path = os.path.join(extract_path, rel_path)
    mic_locs[2, :] = ast.literal_eval(row["room_dimension"])[2] / 2

    waveform, sr = torchaudio.load(wav_path)
    waveform = waveform.numpy()

    # Referans mikrofon olarak 7. indeksi (merkez mikrofon) kullanıyoruz
    ref_idx = 7
    estimated_angle, spatial_spectrum, grid_angles = estimate_azimuth(waveform, mic_locs, sr, ref_idx)

    # Dairesel Hata Hesaplama
    raw_error = abs(true_angle - estimated_angle)
    error = min(raw_error, 360 - raw_error)

    results.append({
        "method": "gcc-phat",
        "file": rel_path,
        "true_angle": true_angle,
        "estimated_angle": estimated_angle,
        "angular_error": error
    })

    plot_data.append({
        "spec": spatial_spectrum,
        "true": true_angle,
        "est": estimated_angle,
        "error": error
    })

    print(f"[{i+1}/2] İşlendi: {rel_path.split('/')[-1]} | Gerçek: {true_angle:.1f}° | Tahmin: {estimated_angle:.1f}°")

# ==========================================
# 4. GRAFİK ÇİZİMİNİ TETİKLEME
# ==========================================
if len(plot_data) == 2:
    if plot_data[0]["error"] < plot_data[1]["error"]:
        idx_success, idx_fail = 0, 1
    else:
        idx_success, idx_fail = 1, 0

    save_combined_spectrum_plot(
        grid_angles=grid_angles,
        spec_success=plot_data[idx_success]["spec"],
        true_success=plot_data[idx_success]["true"],
        est_success=plot_data[idx_success]["est"],
        spec_fail=plot_data[idx_fail]["spec"],
        true_fail=plot_data[idx_fail]["true"],
        est_fail=plot_data[idx_fail]["est"],
        filename=os.path.join(plots_dir, "gcc_phat_karsilastirma.png")
    )

# ==========================================
# 5. CSV KAYDETME
# ==========================================
results_df = pd.DataFrame(results)
results_df.to_csv("gcc-phat-results.csv", index=False)
print("✅ İşlemler tamamlandı, CSV ve GCC-PHAT grafiği başarıyla kaydedildi.")

GCC-PHAT İçin İşlenecek Dosya Sayısı: 2
[1/2] İşlendi: sample_00002.wav | Gerçek: 180.0° | Tahmin: 180.0°
[2/2] İşlendi: sample_00003.wav | Gerçek: 128.0° | Tahmin: 308.4°
✅ Karşılaştırmalı grafik başarıyla kaydedildi: /content/gcc_phat_karsilastirma.png
✅ İşlemler tamamlandı, CSV ve GCC-PHAT grafiği başarıyla kaydedildi.
